# The Engagement Financial Lifecycle: A Walkthrough

*Written from the seat of an Engagement Finance Associate supporting engagement leaders on a
mid-size professional-services practice's Government & Public Services account.*

Every engagement in this book of business is a services contract billed to a government or
public-sector client -- not a product line with a P&L. The financial discipline that matters
here isn't "is the company profitable," it's narrower and more operational: **is this specific
engagement staying within its authorized budget, is the time and expense we're tracking actually
accurate, and is everything we've earned actually getting billed?**

That's the job. This notebook walks through the full lifecycle the way I'd actually work it,
engagement by engagement, from portfolio setup through the exception report I'd send an
engagement leader on a Friday afternoon.

> **All data in this notebook is synthetic**, generated by this repository's Python package for
> portfolio/demonstration purposes. No real client, engagement, or financial figures are
> represented. Client names, engagement names, and dollar figures are fictional.

In [1]:
import pandas as pd

from engagement_finance_toolkit.engagement_setup import generate_portfolio, AS_OF_DATE, ContractType
from engagement_finance_toolkit.time_expense_tracking import (
    generate_timesheets, generate_expenses, timesheets_to_dataframe, expenses_to_dataframe,
)
from engagement_finance_toolkit.budget_actual import build_budget_actual_by_role_month, build_engagement_summary
from engagement_finance_toolkit.invoicing import generate_invoices, reconcile_invoice_to_source, line_items_to_dataframe
from engagement_finance_toolkit.discrepancy_detection import run_all_detectors, build_exception_report, build_leader_summary
from engagement_finance_toolkit.reporting import build_workbook

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print(f"As-of date for this analysis: {AS_OF_DATE.isoformat()}")

As-of date for this analysis: 2025-08-15


## 1. Setting Up the Engagement Portfolio

Before I can track anything, I need to know what I'm tracking against. Every engagement gets set
up with a **contract type**, because that single field determines how the engagement gets
billed, how risk is shared with the client, and what "over budget" even means:

- **Time & Materials (T&M)** -- we bill actual hours at agreed labor-category rates, up to a
  not-to-exceed (NTE) ceiling. The client bears less cost risk; we bear the risk of hours
  creeping past the ceiling without authorization.
- **Fixed-Fee** -- the client pays a set price for defined deliverables (milestones), regardless
  of how many hours it actually takes us. We bear the delivery-cost risk; going over our
  internal hour budget doesn't get us paid more.
- **Cost-Plus-Fixed-Fee** -- common on federal work. We're reimbursed for allowable costs
  (labor at cost rates, not commercial bill rates) plus a negotiated fee. The ceiling here is
  priced in *cost* terms, not bill-rate terms -- a distinction that matters a lot once we get to
  budget tracking below.

Every engagement also gets a **budget by labor category** (Partner / Manager / Consultant /
Associate), because budgets that aren't broken out by role can't actually be managed -- a team
that's over budget on Partner hours and under on Associate hours nets out to "on budget" on
paper while masking a real staffing mix problem.

In [2]:
engagements, staff = generate_portfolio()

mix = pd.Series([e.contract_type.value for e in engagements]).value_counts()
print(f"{len(engagements)} engagements across {len(staff)} staff\n")
print("Contract type mix:")
print(mix.to_string())

portfolio_df = pd.DataFrame(
    [
        {
            "Engagement ID": e.engagement_id,
            "Client": e.client_name,
            "Engagement": e.engagement_name,
            "Contract Type": e.contract_type.value,
            "Leader": e.engagement_leader,
            "Start": e.start_date,
            "End": e.end_date,
            "Budgeted Hours": e.total_budget_hours,
            "Budgeted $": e.total_budget_dollars,
            "Has Change Order": len(e.change_orders) > 0,
        }
        for e in engagements
    ]
)
portfolio_df.head(8)

18 engagements across 26 staff

Contract type mix:
Time & Materials       7
Cost-Plus-Fixed-Fee    6
Fixed-Fee              5


,Engagement ID,Client,Engagement,Contract Type,Leader,Start,End,Budgeted Hours,Budgeted $,Has Change Order
0,ENG-001,Commonwealth Department of Transportation,Highway Asset Management Program Support,Fixed-Fee,M. Alvarez,2024-10-23,2025-05-18,"2,104.00","449,785.00",True
1,ENG-002,Bureau of Health & Human Services,Medicaid Eligibility System Modernization PMO,Fixed-Fee,R. Chen,2024-11-06,2025-07-09,"1,159.00","246,725.00",False
2,ENG-003,Federal Logistics & Readiness Agency,Logistics Readiness ERP Modernization PMO,Cost-Plus-Fixed-Fee,M. Alvarez,2025-02-13,2025-10-22,"1,277.00","272,510.00",False
3,ENG-004,Ashford County Government,County Financial Systems Upgrade Support,Cost-Plus-Fixed-Fee,M. Alvarez,2024-11-02,2025-11-14,"2,894.00","614,015.00",True
4,ENG-005,Metro Regional Transit Authority,Capital Projects Cost Control Advisory,Fixed-Fee,R. Chen,2025-03-02,2025-11-10,"2,810.00","593,475.00",False
5,ENG-006,Commonwealth Department of Transportation,Capital Program Financial Advisory,Cost-Plus-Fixed-Fee,R. Chen,2024-12-05,2026-05-06,"4,106.00","870,360.00",False
6,ENG-007,Bureau of Health & Human Services,Provider Payment Integrity Review,Fixed-Fee,R. Chen,2025-01-21,2025-04-22,236.00,"50,595.00",True
7,ENG-008,Federal Logistics & Readiness Agency,Supply Chain Cost Accounting Advisory,Time & Materials,R. Chen,2025-04-29,2026-06-22,"1,194.00","256,650.00",True


## 2. Time & Expense: Where the Numbers Actually Come From

Every downstream number in this notebook -- budget-to-actual, invoices, realization -- is built
on top of raw timesheet and expense entries logged by the team. That means the *quality* of
those entries sets a ceiling on the quality of everything else. A missing task code, a rate
that doesn't match the person's actual labor category, an entry logged two months late, time
that's sat unapproved for weeks -- these aren't cosmetic problems. Each one either blocks
legitimate revenue from being billed, or lets bad data flow straight into a client invoice.

I'm generating a realistic, light error rate here (~4% of entries) -- not because I'm trying to
make the data look messy, but because a *zero*-error timesheet population would make the rest of
this notebook pointless. In real life, this is exactly the level of noise an associate reviews
every billing cycle.

In [3]:
timesheet_entries = generate_timesheets(engagements, staff)
expense_entries = generate_expenses(engagements, staff)
timesheet_df = timesheets_to_dataframe(timesheet_entries)
expense_df = expenses_to_dataframe(expense_entries)

print(f"Timesheet entries: {len(timesheet_df):,}  ({timesheet_df['hours'].sum():,.0f} total hours)")
print(f"Expense entries:   {len(expense_df):,}  (${expense_df['amount'].sum():,.2f} total)")
print()
print(f"Missing task code:   {timesheet_df['task_code'].isna().mean():.1%}")
print(f"Unapproved entries:  {(~timesheet_df['approved']).mean():.1%}")
lag = (timesheet_df['entered_date'] - timesheet_df['work_date']).apply(lambda d: d.days)
print(f"Entries logged >21 days after the work date: {(lag > 21).mean():.1%}")

Timesheet entries: 4,863  (24,154 total hours)
Expense entries:   395  ($72,987.24 total)

Missing task code:   1.0%
Unapproved entries:  1.2%
Entries logged >21 days after the work date: 1.0%


## 3. Budget vs. Actual: Keeping the Team Honest About Pace

This is the monthly heartbeat of the job. For every engagement, by role, by month, I compare
**budgeted hours to actual hours** (utilization) and I track a **rate realization** figure --
the standard billing value the timesheet system actually captured, versus what it should have
captured if everyone logged time under the correct labor category. Rate realization is a pure
data-quality signal: it only moves when someone's hours got coded at the wrong rate.

At the engagement level, I don't just look at "% of budget spent" in isolation -- a team that's
50% through its hours at the engagement's midpoint is fine; the same 50% one month after
kickoff is a real problem. So I compare **% of budget consumed** against **% of the planned
staffing curve elapsed** (the same ramp-up/steady-state/wind-down shape the plan itself was
built on) -- classic earned-value logic. That comparison is what actually separates "on track"
from "trending over budget" from "behind pace."</br>

One nuance that bit me the first time I built this: for **cost-plus** engagements, the contract
ceiling is priced in *cost* terms (labor at cost rates + a fee), not commercial bill rates. If I
compared bill-rate-valued actuals against that cost-based ceiling, every cost-plus engagement
would look like it was blowing through its ceiling almost immediately -- which isn't real, it's
just a unit mismatch. So ceiling consumption is measured on whichever basis actually matches how
that contract type's ceiling was set.

In [4]:
budget_actual_df = build_budget_actual_by_role_month(engagements, timesheet_df)
engagement_summary_df = build_engagement_summary(engagements, timesheet_df)

print("Status across the portfolio:")
print(engagement_summary_df["status"].value_counts().to_string())
print()

cols = ["engagement_id", "contract_type", "budgeted_hours", "actual_hours_to_date",
        "schedule_pct_elapsed", "budget_pct_consumed", "schedule_budget_variance", "status"]
engagement_summary_df[cols].sort_values("schedule_budget_variance", ascending=False).head(6)

Status across the portfolio:
status
Behind Pace / Under-Utilized    7
On Track                        7
Trending Over Budget            4



,engagement_id,contract_type,budgeted_hours,actual_hours_to_date,schedule_pct_elapsed,budget_pct_consumed,schedule_budget_variance,status
9,ENG-010,Time & Materials,"2,748.00","2,484.90",0.28,0.90,0.62,Trending Over Budget
7,ENG-008,Time & Materials,"1,194.00","1,032.00",0.28,0.86,0.58,Trending Over Budget
10,ENG-011,Time & Materials,"2,213.00","1,304.10",0.13,0.59,0.46,Trending Over Budget
14,ENG-015,Cost-Plus-Fixed-Fee,"2,152.00","1,917.40",0.69,0.89,0.20,Trending Over Budget
4,ENG-005,Fixed-Fee,"2,810.00","2,385.50",0.72,0.85,0.13,On Track
11,ENG-012,Time & Materials,"1,390.00","1,231.30",0.76,0.89,0.12,On Track


## 4. Invoicing: Turning Approved Time Into Cash

Draft invoices get built differently depending on contract type -- T&M and cost-plus bill off
reconciled actuals for each fully-closed billing month (the current, still-open month isn't
billed yet, same as a real firm's billing cadence); fixed-fee bills off the milestone schedule
as milestones are reached.

Two things are deliberately excluded from every draft invoice, by construction: a timesheet
entry **missing its task code** (it can't be routed to a billing category) and one that's
**unapproved** (nothing goes on a client invoice without a manager's sign-off). If those don't
get corrected, that time ages into the "unbilled time" exception you'll see in the next section
-- which is exactly the failure mode a real billing process breakdown looks like.

Every invoice traces back to the specific timesheet/expense rows it was built from, so I can
independently recompute each invoice's total straight from source data and confirm nothing was
dropped or double-counted -- this is the reconciliation step I'd actually do before an invoice
goes out the door.

In [5]:
invoices = generate_invoices(engagements, timesheet_df, expense_df)

reconciliation_results = [reconcile_invoice_to_source(inv, timesheet_df, expense_df) for inv in invoices]
failures = [r for r in reconciliation_results if not r["ties_out"]]
print(f"Draft invoices generated: {len(invoices)}")
print(f"Total draft invoice value: ${sum(inv.total_amount for inv in invoices):,.2f}")
print(f"Invoices that failed reconciliation to source data: {len(failures)}")

sample_invoice = max(invoices, key=lambda inv: inv.total_amount)
print(f"\nSample invoice {sample_invoice.invoice_id} ({sample_invoice.contract_type}), "
      f"{sample_invoice.period_start} to {sample_invoice.period_end}:")
line_items_to_dataframe([sample_invoice])[["description", "category", "hours", "rate", "amount"]]

Draft invoices generated: 99
Total draft invoice value: $3,952,442.66
Invoices that failed reconciliation to source data: 0

Sample invoice INV-ENG-010-202507 (Time & Materials), 2025-07-01 to 2025-07-31:


,description,category,hours,rate,amount
0,Professional Services -- Partner,Labor,54.50,385.00,"20,982.50"
1,Professional Services -- Manager,Labor,141.90,275.00,"39,022.50"
2,Professional Services -- Consultant,Labor,377.10,210.00,"79,191.00"
3,Professional Services -- Associate,Labor,287.00,155.00,"44,485.00"
4,Reimbursable Expenses -- Ground Transport,Expense,NaN,NaN,89.73


## 5. Data Quality & Discrepancy Detection: The Real Job

This is the part of the role that actually matters most, and it's the reason the rest of this
notebook exists -- everything above is infrastructure for this step. Every detector below works
directly off the raw timesheet, expense, invoice, and budget data; nothing is pre-flagged by the
generator. That's deliberate: an associate doesn't get handed a list of "here's what's wrong,"
they have to find it in the export.

Here's what I'm checking for, and why each one matters on a services engagement specifically
(not generic FP&A):

| Category | Why it matters |
|---|---|
| **Missing Task Code** | Blocks that time from ever reaching an invoice until someone fixes it. Silent revenue leakage if it's never caught. |
| **Missing Approval** | Nothing should be billable -- or paid out, for expenses -- without a manager's sign-off. An unapproved entry sitting for months is a control failure, not an oversight. |
| **Rate Mismatch** | Time logged under the wrong labor category distorts realization and, if it ever reached an invoice, would be a contract-compliance problem (billing the wrong rate-card line to a government client). |
| **Out-of-Period Entry** | Time logged long after the work was done makes every prior month's budget-to-actual reporting unreliable in the periods it *should* have landed in. |
| **Unbilled Time / Expense Aging** | Clean, approved, billable work that's sat past a normal billing cycle without being invoiced. Left alone, this is straight revenue leakage. |
| **Budget Overrun (without a Change Order)** | The highest-severity finding here. Spend has exceeded the engagement's authorized ceiling with no client-approved change order on file -- unrecoverable cost exposure if it's not caught and escalated. |

Findings are scored by severity and rolled up into exactly the kind of prioritized exception
report I'd actually send an engagement leader.

In [6]:
findings = run_all_detectors(engagements, timesheet_df, expense_df, invoices, engagement_summary_df)
exception_report_df = build_exception_report(findings)
leader_summary_df = build_leader_summary(exception_report_df)

print(f"Total open exceptions: {len(exception_report_df)}")
print(exception_report_df["severity"].value_counts().to_string())
print()
print("By category:")
print(exception_report_df["category"].value_counts().to_string())

Total open exceptions: 64
severity
Medium    28
High      19
Low       17

By category:
category
Missing Approval              17
Missing Task Code             14
Rate Mismatch                 14
Out-of-Period Entry           14
Unbilled Time Aging            2
Unbilled Expense Aging         2
Approaching Budget Ceiling     1


In [7]:
# The top of the escalation email: highest severity, highest dollar impact first.
exception_report_df[["engagement_id", "category", "severity", "dollar_impact", "description"]].head(10)

,engagement_id,category,severity,dollar_impact,description
0,ENG-009,Unbilled Time Aging,High,"79,722.00","$79,722.00 of approved, billable time (369.2 h..."
1,ENG-018,Unbilled Time Aging,High,"20,410.50","$20,410.50 of approved, billable time (95.2 hr..."
2,ENG-006,Missing Approval,High,"6,656.00","5 time entries are unapproved, oldest 205 days..."
3,ENG-012,Missing Approval,High,"5,623.00","6 time entries are unapproved, oldest 127 days..."
4,ENG-010,Missing Approval,High,"5,460.00","5 time entries are unapproved, oldest 77 days ..."
5,ENG-009,Missing Approval,High,"5,272.00","5 time entries are unapproved, oldest 247 days..."
6,ENG-015,Missing Approval,High,"5,127.50","4 time entries are unapproved, oldest 200 days..."
7,ENG-011,Missing Approval,High,"3,861.50","4 time entries are unapproved, oldest 74 days ..."
8,ENG-003,Missing Approval,High,"3,148.50","4 time entries are unapproved, oldest 51 days ..."
9,ENG-016,Missing Approval,High,"2,734.00","3 time entries are unapproved, oldest 163 days..."


In [8]:
# Rolled up by engagement leader -- what actually goes at the top of the message.
leader_summary_df

,engagement_leader,open_exceptions,high_severity,medium_severity,low_severity,total_dollar_impact,engagements_affected
3,R. Chen,28,7,14,7,"80,911.50",7
1,J. Patel,8,4,2,2,"38,263.62",3
0,D. Okoro,10,3,4,3,"112,927.06",2
2,M. Alvarez,10,3,4,3,"28,838.50",3
4,S. Whitfield,8,2,4,2,"78,698.14",2


## 6. Rolling It Up: The Portfolio Dashboard

The last step is turning all of this into something an engagement leader (or a practice
leadership team looking across the whole portfolio) can actually open and use: a formula-driven
Excel workbook, not a static snapshot. Every KPI, utilization figure, and ceiling-consumption
percentage in the workbook is a live formula referencing the source tabs -- if the underlying
budgeted or actual figures changed, the workbook would recalculate correctly, the same way a
real financial model should.

The workbook has five tabs: **Portfolio Dashboard** (KPI tiles + native charts), **Engagement
Summary**, **Budget vs Actual**, a worked **Sample Invoice**, and the **Discrepancy Report**
with conditional formatting by severity.

In [9]:
output_path = build_workbook(
    engagements,
    budget_actual_df,
    engagement_summary_df,
    invoices,
    exception_report_df,
    leader_summary_df,
    output_path="../output/engagement_finance_toolkit.xlsx",
)
print(f"Workbook written to: {output_path}")

Workbook written to: ../output/engagement_finance_toolkit.xlsx


## Wrapping Up

Walking back through this: the portfolio started as 18 budgets and a staffing plan. By the time
it goes through time & expense tracking, budget-to-actual, invoicing, and discrepancy detection,
what comes out the other end is a prioritized list of exactly what needs an engagement leader's
attention this week -- unbilled time that's aging, approvals that haven't happened, a couple of
engagements approaching their ceiling, and (in a less lucky data pull) an unauthorized overrun
that needs a change order yesterday.

That loop -- catch it in the data before it becomes a client-facing or contract-compliance
problem -- is the actual job. Everything else in this toolkit exists to make that loop possible.

> Reminder: every number in this notebook is synthetic, generated for portfolio/demonstration
> purposes only. No real engagement, client, or financial data is represented.